# CDOT 2.0 — Statistical Analysis

Run this notebook after data collection is complete (N ≥ 40).

**Pre-registered hypotheses tested here:**
- H1: Treatment > Control A (C scores)
- H2: T1 < T2 within treatment sessions
- H3: Treatment A > Control B (CLIS specificity)
- H4: More anomalies with primary investigator
- H5: Intervention types differentially affect dimensions

**Bonferroni-corrected α = 0.0125 for 4 primary tests**

---

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import pingouin as pg
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT = Path('..')
DATA = ROOT / 'data'

# Load data
sessions = pd.read_csv(DATA / 'scores' / 'session_master_log.csv')
sessions = sessions.dropna(subset=['date'])
anomalies = pd.read_csv(DATA / 'anomalies' / 'behavioral_anomaly_log.csv')
anomalies = anomalies.dropna(subset=['session_id'])

# Convert scores to numeric
score_cols = [c for c in sessions.columns if c.startswith('t') and '_' in c[1:]]
for col in score_cols:
    sessions[col] = pd.to_numeric(sessions[col], errors='coerce')
sessions['c_formula_normalized'] = pd.to_numeric(sessions['c_formula_normalized'], errors='coerce')

print(f'Total sessions loaded: {len(sessions)}')
print(f'Total anomalies loaded: {len(anomalies)}')
print(f'\nConditions: {sessions["condition"].value_counts().to_dict()}')

## H1: Treatment > Control A

In [ ]:
treatment = sessions[sessions['condition'].str.startswith('Treatment', na=False)]['c_formula_normalized'].dropna()
control_a = sessions[sessions['condition'] == 'Control A']['c_formula_normalized'].dropna()

if len(treatment) > 0 and len(control_a) > 0:
    # Normality check
    print('=== H1: Treatment vs Control A ===')
    print(f'\nTreatment: n={len(treatment)}, M={treatment.mean():.2f}, SD={treatment.std():.2f}')
    print(f'Control A: n={len(control_a)}, M={control_a.mean():.2f}, SD={control_a.std():.2f}')
    
    # Shapiro-Wilk normality test
    if len(treatment) >= 3:
        w_t, p_t = stats.shapiro(treatment)
        w_c, p_c = stats.shapiro(control_a) if len(control_a) >= 3 else (0, 0)
        print(f'\nNormality (Shapiro-Wilk): Treatment p={p_t:.4f}, Control p={p_c:.4f}')
    
    # Main test
    if len(treatment) >= 3 and len(control_a) >= 3:
        t_stat, p_val = stats.ttest_ind(treatment, control_a)
        d = pg.compute_effsize(treatment, control_a, eftype='cohen')
        
        # Mann-Whitney as non-parametric alternative
        u_stat, p_mw = stats.mannwhitneyu(treatment, control_a, alternative='greater')
        
        print(f'\nIndependent t-test: t={t_stat:.3f}, p={p_val:.4f}')
        print(f'Mann-Whitney U: U={u_stat:.1f}, p={p_mw:.4f}')
        print(f'Cohen\'s d: {d:.3f}')
        print(f'\nBonferroni-corrected α: 0.0125')
        print(f'Significant: {"YES" if p_val < 0.0125 else "NO"}')
        print(f'Effect size: {"Large" if abs(d) >= 0.8 else "Medium" if abs(d) >= 0.5 else "Small"}')
else:
    print('Insufficient data for H1 test. Need treatment AND control sessions.')

## H2: Within-Session T1 → T2 Change

In [ ]:
treatment_sessions = sessions[sessions['condition'].str.startswith('Treatment', na=False)].copy()

# Calculate T1 and T2 totals from dimension scores
for prefix in ['t1', 't2']:
    dim_cols = [f'{prefix}_{d}' for d in ['R', 'M', 'T', 'P', 'D']]
    available = [c for c in dim_cols if c in treatment_sessions.columns]
    if available:
        treatment_sessions[f'{prefix}_total'] = treatment_sessions[available].sum(axis=1)

if 't1_total' in treatment_sessions.columns and 't2_total' in treatment_sessions.columns:
    paired = treatment_sessions[['t1_total', 't2_total']].dropna()
    
    if len(paired) >= 3:
        print('=== H2: Within-Session T1 → T2 ===')
        print(f'\nN pairs: {len(paired)}')
        print(f'T1 mean: {paired["t1_total"].mean():.2f}')
        print(f'T2 mean: {paired["t2_total"].mean():.2f}')
        print(f'Mean change: {(paired["t2_total"] - paired["t1_total"]).mean():.2f}')
        
        t_stat, p_val = stats.ttest_rel(paired['t2_total'], paired['t1_total'])
        d = pg.compute_effsize(paired['t2_total'], paired['t1_total'], paired=True, eftype='cohen')
        
        print(f'\nPaired t-test: t={t_stat:.3f}, p={p_val:.4f}')
        print(f'Cohen\'s d: {d:.3f}')
        print(f'Bonferroni-corrected α: 0.0125')
        print(f'Significant: {"YES" if p_val < 0.0125 else "NO"}')
    else:
        print('Insufficient paired data for H2.')
else:
    print('T1/T2 columns not found. Ensure CDOT Mini scores are entered.')

## H3: CLIS Specificity (Treatment A vs Control B)

In [ ]:
treat_a = sessions[sessions['condition'] == 'Treatment A']['c_formula_normalized'].dropna()
control_b = sessions[sessions['condition'] == 'Control B']['c_formula_normalized'].dropna()

if len(treat_a) > 0 and len(control_b) > 0:
    print('=== H3: CLIS Specificity ===')
    print(f'\nTreatment A (Primary): n={len(treat_a)}, M={treat_a.mean():.2f}, SD={treat_a.std():.2f}')
    print(f'Control B (Blind):     n={len(control_b)}, M={control_b.mean():.2f}, SD={control_b.std():.2f}')
    
    t_stat, p_val = stats.ttest_ind(treat_a, control_b)
    d = pg.compute_effsize(treat_a, control_b, eftype='cohen')
    
    print(f'\nIndependent t-test: t={t_stat:.3f}, p={p_val:.4f}')
    print(f'Cohen\'s d: {d:.3f}')
    print(f'Bonferroni-corrected α: 0.0125')
    print(f'Significant: {"YES" if p_val < 0.0125 else "NO"}')
    
    if p_val < 0.0125:
        print('\n→ CLIS SPECIFICITY SUPPORTED: Primary investigator produces effects beyond protocol alone.')
    else:
        print('\n→ CLIS specificity NOT supported: Effects appear protocol-driven (still valuable — scalable).')
else:
    print('Need both Treatment A and Control B data for CLIS specificity test.')

## H4: Anomaly Frequency by Investigator

In [ ]:
# Count sessions with any anomaly
sessions['has_anomaly'] = pd.to_numeric(sessions['anomaly_count'], errors='coerce').fillna(0) > 0
sessions['is_blind'] = sessions['condition'] == 'Control B'

primary = sessions[~sessions['is_blind']]
blind = sessions[sessions['is_blind']]

if len(primary) > 0 and len(blind) > 0:
    print('=== H4: Anomaly Frequency ===')
    
    a_primary = primary['has_anomaly'].sum()
    n_primary = len(primary)
    a_blind = blind['has_anomaly'].sum()
    n_blind = len(blind)
    
    print(f'\nPrimary: {a_primary}/{n_primary} sessions with anomalies ({a_primary/n_primary*100:.0f}%)')
    print(f'Blind:   {a_blind}/{n_blind} sessions with anomalies ({a_blind/n_blind*100:.0f}%)')
    
    # Fisher's exact test
    table = [[a_primary, n_primary - a_primary],
             [a_blind, n_blind - a_blind]]
    odds_ratio, p_val = stats.fisher_exact(table)
    
    print(f'\nFisher\'s exact test: OR={odds_ratio:.2f}, p={p_val:.4f}')
    print(f'Bonferroni-corrected α: 0.0125')
    print(f'Significant: {"YES" if p_val < 0.0125 else "NO"}')
else:
    print('Need both primary and blind operator data.')

## H5: Intervention Type × Dimension Interaction (Exploratory)

In [ ]:
treat_b = sessions[sessions['condition'] == 'Treatment B'].copy()

if len(treat_b) >= 5:
    print('=== H5: Intervention Type × Dimension (Exploratory) ===')
    
    # Show mean dimension scores by intervention type
    dims = {'t3_R': 'Relational', 't3_M': 'Meta', 't3_T': 'Traversal', 't3_P': 'Polarity', 't3_D': 'Damping'}
    
    for dim_col, dim_name in dims.items():
        if dim_col in treat_b.columns and 'intervention_type' in treat_b.columns:
            grouped = treat_b.groupby('intervention_type')[dim_col].agg(['mean', 'std', 'count'])
            if len(grouped) > 0:
                print(f'\n{dim_name} by intervention type:')
                print(grouped.round(2).to_string())
    
    # ANOVA if enough data
    if 'intervention_type' in treat_b.columns:
        types = treat_b['intervention_type'].dropna().unique()
        if len(types) >= 2:
            for dim_col, dim_name in dims.items():
                groups = [treat_b[treat_b['intervention_type'] == t][dim_col].dropna() for t in types]
                groups = [g for g in groups if len(g) >= 2]
                if len(groups) >= 2:
                    f_stat, p_val = stats.f_oneway(*groups)
                    print(f'\n{dim_name} ANOVA: F={f_stat:.3f}, p={p_val:.4f}')
else:
    print('Need at least 5 Treatment B sessions for intervention comparison.')

## CDOT Reliability (Cronbach's Alpha)

In [ ]:
def cronbachs_alpha(items_df):
    """Calculate Cronbach's alpha for a set of items."""
    items = items_df.dropna()
    if len(items) < 3:
        return None
    k = items.shape[1]
    item_vars = items.var(axis=0, ddof=1)
    total_var = items.sum(axis=1).var(ddof=1)
    if total_var == 0:
        return None
    alpha = (k / (k - 1)) * (1 - item_vars.sum() / total_var)
    return alpha

print('=== CDOT Reliability ===')
print('(Requires CDOT Full item-level data in data/scores/)\n')

# Check for item-level data files
score_files = list((DATA / 'scores').glob('cdot_full_*.csv'))
if score_files:
    all_items = pd.concat([pd.read_csv(f) for f in score_files])
    all_items['rating_0_10'] = pd.to_numeric(all_items['rating_0_10'], errors='coerce')
    
    for dim in ['R', 'M', 'T', 'P', 'D']:
        dim_items = all_items[all_items['dimension'] == dim]
        if len(dim_items) > 0:
            pivot = dim_items.pivot_table(index='session_id', columns='item_number', values='rating_0_10')
            alpha = cronbachs_alpha(pivot)
            if alpha is not None:
                status = '✓' if alpha >= 0.70 else '⚠'
                print(f'{status} {dim}: α = {alpha:.3f}')
else:
    print('No item-level CDOT data found yet.')
    print('Save completed CDOT Full assessments to data/scores/cdot_full_SESSION_XXX.csv')

## Summary Table (Publication-Ready)

In [ ]:
def generate_results_summary(sessions):
    """Generate a publication-ready results summary table."""
    if len(sessions) < 10:
        print('Need at least 10 sessions for meaningful summary.')
        return
    
    print('\n' + '='*70)
    print('CDOT 2.0 RESULTS SUMMARY')
    print('='*70)
    
    conditions = sessions['condition'].unique()
    
    print(f'\n{"Condition":<20} {"N":>4} {"Mean C":>8} {"SD":>8} {"Anomalies":>10}')
    print('-'*55)
    
    for cond in sorted(conditions):
        subset = sessions[sessions['condition'] == cond]
        n = len(subset)
        c_scores = subset['c_formula_normalized'].dropna()
        mean_c = c_scores.mean() if len(c_scores) > 0 else 0
        sd_c = c_scores.std() if len(c_scores) > 1 else 0
        anom = pd.to_numeric(subset['anomaly_count'], errors='coerce').sum()
        
        print(f'{cond:<20} {n:>4} {mean_c:>8.2f} {sd_c:>8.2f} {int(anom):>10}')
    
    print('-'*55)
    total_c = sessions['c_formula_normalized'].dropna()
    total_anom = pd.to_numeric(sessions['anomaly_count'], errors='coerce').sum()
    print(f'{"TOTAL":<20} {len(sessions):>4} {total_c.mean():>8.2f} {total_c.std():>8.2f} {int(total_anom):>10}')
    
    print('\n' + '='*70)

generate_results_summary(sessions)